#### 0. Change directory

In [1]:
# Reference: https://stackoverflow.com/questions/15514593/importerror-no-module-named-when-trying-to-run-python-script/15622021#15622021
# Here choosing the folder that stores all the modelling modules
import sys
sys.path.append(r'S:\\ML_original_uncertainty_esk\\LISFLOOD_FP\\Modelling')

#### 1. Packages

In [2]:
from folder import *
from randomisation import random_values_generation
from dataPreparation import download_lidar, \
                            padding_combination, \
                            dem_raster_reference, \
                            terrain_shading             # For lidar downloading, padding calculation, dem reference generation, and terrain shading

from transformation import center_calculation, get_url, get_tile_files, \
                           MyPool, transformation_performance, transformation_parallelism # For transformation using multiprocessing

from rasterSimulation import raster_generation

from floodModel import tide_flow_data, bdy_generation, tideboundary_points, flood_simulation

# For creating plot
import matplotlib.pyplot

# For untransformation
from untransformation import untransformation_parallelism

import multiprocessing
from functools import partial


# For terrain shading
import os
import rioxarray as rxr
import xarray as xr
from osgeo import gdal

# For writing dictionary
import json

#### 2. Necessary variables

In [3]:
# 20-m resolution
# # Construct a list of boundary
# resolution = 20
# number_pixel_x = 16 * 21
# number_pixel_y = 16 * 14

# xmin = 1767790
# ymin = 5471400

In [4]:
# 10-m resolution
# Construct a list of boundary
resolution = 10
number_pixel_x = 16 * 50
number_pixel_y = 16 * 40

xmin = 1929005
ymin = 5629885

In [5]:
# # 10-m resolution
# # Construct a list of boundary
# resolution = 10
# number_pixel_x = 16 * 44
# number_pixel_y = 16 * 65

# xmin = 1929000
# ymin = 5626700

In [6]:
# 5-m resolution
# # Construct a list of boundary
# resolution = 5
# number_pixel_x = 16 * 84
# number_pixel_y = 16 * 56

# xmin = 1767790
# ymin = 5471400

In [7]:
# 2-m resolution
# # Construct a list of boundary
# resolution = 2
# number_pixel_x = 16 * 210
# number_pixel_y = 16 * 140

# xmin = 1767790
# ymin = 5471400

##### 2.2. Other basic variables

In [8]:
xmax = xmin + resolution * number_pixel_x
ymax = ymin + resolution * number_pixel_y

# Boundary for the area of interest
boundary_1 = [xmin, ymin, xmax, ymax]

# Padding boundary for DEMs (use this boundary to create padding)
addition_2 = 16 * 3
boundary_2 = padding_combination(boundary_1, addition_2)

# Boundary for tiles (use this boundary to download LiDAR)
addition_3 = 16 * 3
boundary_3 = padding_combination(boundary_2, addition_3)

In [9]:
# LiDAR dataset name
lidar_name = "NZ20_Hawkes"

# # Chunks and processors information (for computer has 8 cores)
# For 10-m resolution
size_of_chunk = 100
size_of_processor = 4
# # For 20-m resolution
# size_of_chunk = 50
# size_of_processor = 2
# # For 2-m resolution
# size_of_chunk = 400
# size_of_processor = 6
# # For 5-m resolution
# size_of_chunk = 200
# size_of_processor = 6

# Flow data
flow_path = fr"S:\\new_version_esk\\version_006\\other_data\\flow.csv"

# Tide data
tide_path = fr"S:\\new_version_esk\\version_006\\other_data\\tide.csv"

# date start and end
date_start = r"2023-02-11 22:00:00"
date_end = r"2023-02-15 15:00:00"

# Tide flow data
tide_flow_data(tide_path, flow_path, date_start, date_end)

# Extracting flowdepth rate (for flowdepth_extraction() function)
flowdepth_rate = 0

# Time to extract water (for water_extraction() function)
extract_wd = "out.max"
extract_wse = "out.mxe"
extract_elev = "elev"

##### 2.3. Preparing flood model parameters

In [10]:
# General parameters
general_para = {
    ## GEOFABRICS
    # Land
    "land": 51559,
    # Bathymetry
    "ocean_contours": 50448,
    
    # Waterways --------
    "drain": 5,
    "stream": 7.5,
    "river": 10,
    
    # Osm
    "osm_id": 109375060,
    "osm_date": "2023-02-02T00:00:00Z",
    
    # River features
    "max_channel_width": 200,
    "min_channel_width": 15,
    "max_bank_height": 4,
    "min_bank_height": 0.5,
    
    # Others
    "network_id": 8021954,
    "area_threshold": 249500000,
    "width_centre_smoothing": 10,
    "river_corridor_width": 300,
    "cross_section_spacing": 10,
    "rec_alignment_tolerance": 65,
    "veg_lidar_classifications_to_keep": [2, 3, 4, 5, 9],
    
    # File
    "network_file": "river_network_esk.geojson",
    
    ## TRANSFORMATION
    'offset_x': 1900000,
    'offset_y': 5600000,
    
    
    ## FLOOD MODEL
    "tide_sampledis": 10, # 10 or 10
    "tide_num_points": 652, # 652 or 13
    "sim_time": 320400,
    
    # SEA FILTER
    'sea_catchment': 0
}

# Write out json
with open(fr"{other_data}/general_parameters.json", "w") as general_parameters_file:
    json.dump(general_para, general_parameters_file)

In [11]:
# Create tide_flow data
tide_flow_data(tide_path, flow_path, date_start, date_end)

# River points
# river_points = (1774477.413, 5471521.558)
# river_points = (1490253.240, 5367720.032)
river_points = (1929029.16, 5633440.46)

# Create BDY file
domain_name = "Esk"
bdy_generation(resolution, domain_name)

# Create tide boundary
tideboundary_points(boundary_1, False)

##### 2.4. Preparing some flood model inputs

In [12]:
%%time
# # Download LiDAR
# download_lidar(boundary_3, lidar_name) # ~ 15 mins

CPU times: total: 0 ns
Wall time: 0 ns


In [13]:
# %%time
# # Reference DEM without padding
# dem_raster_reference(resolution, size_of_chunk, size_of_processor,
#                      boundary_1, lidar_name, "no_padding")

In [14]:
# %%time
# # Reference DEM with padding
# dem_raster_reference(resolution, size_of_chunk, size_of_processor,
#                      boundary_2, lidar_name, "padding")

In [15]:
# %%time
# # DEM for hillshade/terrain shade
# dem_raster_reference(5, 200, 6,
#                      boundary_1, lidar_name, "shading")

In [16]:
# %%time
# # Terrain shading
# terrain_shading(50, 355)

In [17]:
# Calculate coordinates of center point
center_point = center_calculation(True)
center_x = center_point[0]                     # Extract x coordinate of center point
center_y = center_point[1]                     # Extract y coordinate of center point

# Get url list file
url_list_file = get_url(lidar_name)
tile_files = get_tile_files(lidar_name)

##### 2.5. Random transformations

##### 2.5.1. E translation (x)

In [18]:
# # Transformation values
# ran_trans = random_values_generation(
#     1,
#     50,
#     [0, 0],
#     [0, resolution/2],
#     [0, 0],
#     'uniform',
#     True
# )

##### 2.5.2. N translation (y)

In [19]:
# # Transformation values
# ran_trans = random_values_generation(
#     1,
#     50,
#     [0, 0],
#     [0, 0],
#     [0, resolution/2],
#     'uniform',
#     True
# )

##### 2.5.3. N-E translation

In [20]:
# # Transformation values
# ran_trans = random_values_generation(
#     1,
#     50,
#     [0, 0],
#     [0, resolution/2],
#     [0, resolution/2],
#     'uniform',
#     True
# )

##### 2.5.4. Rotation

In [21]:
# # Transformation values
# ran_trans = random_values_generation(
#     1,
#     50,
#     [0, 90],
#     [0, 0],
#     [0, 0],
#     'uniform',
#     True
# )

In [22]:
# Transformation values
ran_trans = random_values_generation(
    1,
    1,
    [0, 0],
    [0, 0],
    [0, 0],
    'uniform',
    True
)

In [23]:
ran_trans

array([[0., 0., 0.]])

##### 2.5.5. N-E translation and rotation

In [24]:
# # Transformation values
# ran_trans = random_values_generation(
#     1,
#     50,
#     [0, 90],
#     [0, resolution/2],
#     [0, resolution/2],
#     'uniform',
#     True
# )

##### 2.6. Only for 2-m resolution (if not skip this part)

In [25]:
# import numpy as np
# ran_trans = np.delete(ran_trans, [3, 4, 12, 40, 41], 0)
# ran_trans = ran_trans[:50]

#### 3. Execution

In [26]:
# %%time
# transformation_parallelism(
#     center_x,
#     center_y,
#     lidar_name,
#     tile_files,
#     30,
#     url_list_file,
#     ran_trans,
#     1
# )

In [27]:
# %%time
# resolution_func = resolution
# chunk_size_func = size_of_chunk
# processor_func = size_of_processor
# padding_func = boundary_2
# lidar_dataset_name = lidar_name
# for ran_trans_i in ran_trans:
#     raster_generation(
#         resolution_func,
#         chunk_size_func,
#         processor_func,
#         padding_func,
#         lidar_dataset_name,
#         center_x, center_y,
#         ran_trans_i
#     )

In [28]:
# %%time
# # DEM CREATION
# # List parameters
# resolution_func = resolution
# chunk_size_func = size_of_chunk
# processor_func = size_of_processor
# padding_func = boundary_2
# lidar_dataset_name = lidar_name

# # Design func
# func = partial(
#     raster_generation,
#     resolution_func,
#     chunk_size_func,
#     processor_func,
#     padding_func,
#     lidar_dataset_name,
#     center_x, center_y
# )

# # Design pool
# pool = MyPool(1)
# pool.map(func, [ran for ran in ran_trans])
# pool.close()
# pool.join()

In [29]:
# %%time
# for ran_trans_i in ran_trans:
#     flood_simulation(
#         domain_name,
#         river_points,
#         center_x, center_y,
#         fr"S:\\new_versions\\data",
#         ran_trans_i
#     )

In [30]:
%%time
# Water depth
untransformation_parallelism(
    extract_wd,
    ran_trans,
    1
)

CPU times: total: 0 ns
Wall time: 59.6 s


In [31]:
%%time
# Water surface elevation
untransformation_parallelism(
    extract_wse,
    ran_trans,
    1
)

CPU times: total: 15.6 ms
Wall time: 1min


In [32]:
%%time
# Elevation
untransformation_parallelism(
    extract_elev,
    ran_trans,
    1
)

CPU times: total: 46.9 ms
Wall time: 1min 1s
